# NumCompute Quickstart

This notebook demonstrates a lightweight end-to-end workflow for the current `RY` branch implementation of `NumCompute`.

It focuses on:

- loading CSV data with `load_csv`
- handling missing values with `SimpleImputer`
- scaling numeric data with `StandardScaler` and `MinMaxScaler`
- encoding categorical data with `OneHotEncoder`
- chaining preprocessing steps with `Pipeline`
- using the current `Pipeline.predict()` interface with a simple model-like step

## 1. Setup imports

The cell below makes sure the repository root is on `sys.path`, so the notebook can be run either from the project root or from a `demo/` folder.

In [67]:
import sys
from pathlib import Path
import numpy as np

# Make the repository root importable
cwd = Path.cwd()
candidate_paths = [cwd, cwd.parent]
for p in candidate_paths:
    if (p / "numcompute").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find the repository root containing the 'numcompute' package.")

print("Repository root:", repo_root)

Repository root: c:\Users\Administrator\Desktop\works\programming-for-ai-group-assignment


In [68]:
from numcompute.io import load_csv
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.pipeline import Pipeline

## 2. Load raw data

In [69]:
data_path = repo_root / "student_performance_data.csv"
X_raw = load_csv(str(data_path), delimiter=",", skip=1)

print("Raw loaded array:")
print(X_raw)
print()
print("Type:", type(X_raw))
print("Shape:", X_raw.shape)

Raw loaded array:
[['1' '19' '12.5' '0.91' '78' '74' 'CS' 'Online' '1']
 ['2' '21' '9' '0.85' '69' '65' 'IT' 'OnCampus' '1']
 ['3' '20' '15' '0.97' '88' '90' 'DS' 'Online' '1']
 ['4' '22' '6.5' '0.72' '55' '58' 'SE' 'Hybrid' '0']
 ['5' '23' '8' '0.8' '61' '63' 'CS' 'OnCampus' '0']
 ['6' '20' '11' '0.89' '73' '70' 'IT' 'Hybrid' '1']
 ['7' '24' '4.5' '0.6' '48' '52' 'DS' 'Online' '0']
 ['8' '21' '13' '0.95' '85' '87' 'CS' 'Hybrid' '1']
 ['9' '19' '7.5' '0.78' '64' '60' 'SE' 'OnCampus' '0']
 ['10' '22' '10.5' '0.88' '72' '76' 'IT' 'Online' '1']
 ['11' '20' '14' '0.96' '90' '92' 'DS' 'Hybrid' '1']
 ['12' '23' '5.5' '0.67' '51' '49' 'CS' 'OnCampus' '0']
 ['13' '21' '9.5' '0.83' '68' '71' 'SE' 'Online' '1']
 ['14' '25' '3' '0.55' '42' '45' 'IT' 'Hybrid' '0']
 ['15' '19' '16' '0.99' '93' '95' 'CS' 'OnCampus' '1']
 ['16' '22' '8.5' '0.81' '66' '64' 'DS' 'Online' '1']
 ['17' '20' 'nan' '0.9' '75' '73' 'IT' 'Hybrid' '1']
 ['18' '24' '6' 'nan' '57' '59' 'SE' 'OnCampus' '0']
 ['19' '21' '12' '0.93

## 3. Create a small numeric example with missing values

This keeps the quickstart simple and directly demonstrates how the preprocessing classes behave.

In [70]:
X_numeric = np.array([
    [20.0, np.nan, 88.0],
    [21.0, 4.0, 91.0],
    [19.0, 2.8, np.nan]
], dtype=float)

print("Original numeric data:")
print(X_numeric)

Original numeric data:
[[20.   nan 88. ]
 [21.   4.  91. ]
 [19.   2.8  nan]]


## 4. Constant imputation

In [71]:
imputer = SimpleImputer(strategy="constant", fill_value=0)
X_imputed = imputer.fit_transform(X_numeric)

print("Imputed numeric data:")
print(X_imputed)

Imputed numeric data:
[[20.   0.  88. ]
 [21.   4.  91. ]
 [19.   2.8  0. ]]


## 5. Standard scaling

In [72]:
scaler = StandardScaler()
X_standard = scaler.fit_transform(X_imputed)

print("Standard-scaled data:")
print(X_standard)
print()
print("Column means learned by scaler:")
print(scaler.mean_)
print("Column scales learned by scaler:")
print(scaler.scale_)

Standard-scaled data:
[[ 0.         -1.35244738  0.67127116]
 [ 1.22474487  1.03422447  0.74234693]
 [-1.22474487  0.31822291 -1.41361808]]

Column means learned by scaler:
[20.          2.26666667 59.66666667]
Column scales learned by scaler:
[ 0.81649658  1.67597401 42.20847729]


## 6. Min-max scaling

In [73]:
minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X_imputed)

print("Min-max scaled data:")
print(X_minmax)
print()
print("Per-feature minimums:")
print(minmax.min_)
print("Per-feature maximums:")
print(minmax.max_)

Min-max scaled data:
[[0.5        0.         0.96703297]
 [1.         1.         1.        ]
 [0.         0.7        0.        ]]

Per-feature minimums:
[19.  0.  0.]
Per-feature maximums:
[21.  4. 91.]


## 7. One-hot encoding for categorical features

In [74]:
X_categorical = np.array([
    ["CS", "Online"],
    ["Math", "Offline"],
    ["CS", "Offline"],
    ["Physics", "Online"]
], dtype=object)

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_categorical)

print("Original categorical data:")
print(X_categorical)
print()
print("Learned categories:")
print(encoder.categories_)
print()
print("One-hot encoded data:")
print(X_encoded)

Original categorical data:
[['CS' 'Online']
 ['Math' 'Offline']
 ['CS' 'Offline']
 ['Physics' 'Online']]

Learned categories:
[array(['CS', 'Math', 'Physics'], dtype=object), array(['Offline', 'Online'], dtype=object)]

One-hot encoded data:
[[1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0.]
 [1. 0. 0. 1. 0.]
 [0. 0. 1. 0. 1.]]


## 8. Use `Pipeline` for preprocessing

The current `Pipeline` implementation supports `fit`, `transform`, and `fit_transform`, so it can chain multiple preprocessors into one reusable workflow.

In [75]:
preprocess_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

X_processed = preprocess_pipe.fit_transform(X_numeric)

print("Pipeline output:")
print(X_processed)

Pipeline output:
[[ 0.         -1.35244738  0.67127116]
 [ 1.22474487  1.03422447  0.74234693]
 [-1.22474487  0.31822291 -1.41361808]]


## 9. Use `Pipeline.predict()` with a simple model-like step

In the current version of `pipeline.py`, the final step can provide `predict()`.  
To match the current implementation, the model-like object below uses `fit(X)` without a `y` argument.

In [76]:
class DummyThresholdModel:
    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self._is_fitted = False

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        # A simple fitted statistic from transformed data
        self.reference_ = np.mean(X[:, -1])
        self._is_fitted = True
        return self

    def predict(self, X):
        if not self._is_fitted:
            raise ValueError("DummyThresholdModel has not been fitted yet.")
        X = np.asarray(X, dtype=float)
        return (X[:, -1] >= self.reference_).astype(int)

model_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler()),
    ("model", DummyThresholdModel())
])

model_pipe.fit(X_numeric)
y_pred = model_pipe.predict(X_numeric)

print("Predicted labels:")
print(y_pred)

Predicted labels:
[1 1 0]


## 10. Summary

This quickstart demonstrated the current branch workflow:

- `load_csv` loads raw delimited data into NumPy arrays
- `SimpleImputer` fills missing values
- `StandardScaler` and `MinMaxScaler` normalize numeric features
- `OneHotEncoder` converts categorical columns into binary features
- `Pipeline` chains preprocessing steps together
- `Pipeline.predict()` can be used when the final step provides a `predict` method